In [ ]:
%run ./00_Mapping_Setup

In [ ]:
%sql
/* ===================================================================================
   METRIC: EMP05 - Contingent Workers in High-Risk Jurisdictions
   
   BUSINESS LOGIC:
   1. Filters HR Enterprise List for 'Contingent' workers.
   2. Filters TD Country Risk Ratings ABAC for 'High' or 'Very High' risk.
   3. Joins HR Data to Cost Center Mapping using CAST(CostCenterID AS INT).
   4. Joins worker's Workcountry to the High-Risk jurisdiction list.
   5. Outputs 'Yes' if any contingent worker in that AU is in a high-risk country.
=================================================================================== */

WITH High_Risk_Countries AS (
    -- Pulls directly from the ABAC schema table
    SELECT DISTINCT 
        UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_ratings_abac
    WHERE TRIM(Final_ABAC_Rating) IN ('High', 'Very High') 
),

Contingent_Workers AS (
    -- Filter HR list for Contingent employees only
    SELECT 
        TRY_CAST(TRIM(CostCenterID) AS INT) AS Cleaned_CC,
        UPPER(TRIM(Workcountry)) AS Work_Country
    FROM hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
    WHERE LOWER(TRIM(WorkerType)) LIKE '%contingent%'
),

Mapped_AUs AS (
    -- Join Mapping -> HR -> Risk Table
    SELECT 
        m.AU_ID AS BusinessID,
        m.AU_Name, 
        
        -- Logic: If the worker's country is in the High Risk list, flag as 1
        CASE 
            WHEN h.CountryName IS NOT NULL THEN 1 
            ELSE 0 
        END AS Has_High_Risk_Worker
        
    FROM vw_cost_center_mapping_bootstrap m
    
    -- LEFT JOIN to HR: Keep all AUs even if they have no workers
    LEFT JOIN Contingent_Workers c 
        ON CAST(m.Cost_Center_ID AS INT) = c.Cleaned_CC
        
    -- LEFT JOIN to Risk: Only matches if country is high risk
    LEFT JOIN High_Risk_Countries h
        ON c.Work_Country = h.CountryName
)

-- Final Output Template
SELECT 
    BusinessID,
    AU_Name,
    'EMP05' AS QuestionID,
    'Applicable' AS Applicability,
    '' AS ApplicabilityRationale,
    
    CASE 
        WHEN SUM(Has_High_Risk_Worker) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response
    
FROM Mapped_AUs
GROUP BY BusinessID, AU_Name
ORDER BY BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EMP05 - Contingent Worker Drill-Down
   Use this to verify exactly WHO is triggering a 'Yes' response.
=================================================================================== */

WITH High_Risk_Countries AS (
    SELECT DISTINCT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_ratings_abac
    WHERE TRIM(Final_ABAC_Rating) IN ('High', 'Very High') 
)

SELECT 
    m.AU_ID AS BusinessID,
    m.AU_Name,
    m.Cost_Center_ID AS Mapped_CC,
    
    -- Worker Details
    cw.FullName,
    cw.WorkerType,
    cw.Work_Country AS Employee_Country,
    
    -- Risk Status Check
    CASE 
        WHEN h.CountryName IS NOT NULL THEN '🚨 HIGH RISK MATCH' 
        ELSE 'Clear / Not High Risk' 
    END AS Risk_Status
    
FROM vw_cost_center_mapping_bootstrap m

-- INNER JOIN: Only show actual contingent workers found in the mapping
INNER JOIN (
    SELECT 
        TRY_CAST(TRIM(CostCenterID) AS INT) AS Cleaned_CC,
        UPPER(TRIM(Workcountry)) AS Work_Country,
        TRIM(WorkerType) AS WorkerType,
        TRIM(FullName) AS FullName
    FROM hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
    WHERE LOWER(TRIM(WorkerType)) LIKE '%contingent%'
) cw ON CAST(m.Cost_Center_ID AS INT) = cw.Cleaned_CC

-- Check against Risk Table
LEFT JOIN High_Risk_Countries h
    ON cw.Work_Country = h.CountryName

ORDER BY Risk_Status ASC, m.AU_ID;